[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | [Tasks](Tasks.md) | [Task 1](Task-1.md) | [Task 2](Task-2.md) | Notebook

# AS Centrality

In [ ]:
name = "YOUR NAME HERE"
date = "MM/DD/YYYY"

In [ ]:
%pip install -q requests pytricia pandas matplotlib pybgpkit-parser

import gc
import math
import time
import shutil
from pathlib import Path
from collections import defaultdict, Counter
import multiprocessing as mp

import requests
import pytricia
import pandas as pd
import matplotlib.pyplot as plt
import pybgpkit_parser as bgpkit

## Task 1: Prepare data

### Task 1.1: Obtain BGP data

In the [BGP assignment](https://github.com/CAIDA/nids-bgp-control-plane), you fetched BGP Routing Information Base (RIB) snapshots from a *single* collector managed by [RouteViews](https://www.routeviews.org/routeviews/). This time, you will fetch the same type of data from *multiple* collectors, managed by [RIPE RIS](https://www.ripe.net/analyse/internet-measurements/routing-information-service-ris/). 

Using data from more collectors allows for a richer view of the Internet's structure at the cost of requiring significantly more computing power. When first writing your code, you can use `COLLECTORS = ["rrc06"]` so that the notebook executes more quickly. When answering the questions for the tasks, use the larger set of data with `COLLECTORS = ["rrc00", "rrc15", "rrc23"]`.

| Collector | Location | File size |
|---|---|---|
| `rrc00` | Amsterdam, NL | ~404 MB |
| `rrc06` | Otemachi, JP | ~41 MB |
| `rrc15` | São Paolo, BR | ~137 MB |
| `rrc23` | Singapore, SG | ~81 MB |

In [ ]:
COLLECTORS = ["rrc06"]                        # Uncomment for testing (41 MB)
# COLLECTORS = ["rrc00", "rrc15", "rrc23"]    # Uncomment for full analysis
SNAPSHOT_DATE = "20260801"                    # YYYYMMDD
SNAPSHOT_HOUR = "0000"                        # 0000, 0800, 1600 UTC

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_rib_file(collector, date=None, hour=None):
    """
    Download a RIB file into data directory if not already cached;
    Return file path.
    """
    date = date or SNAPSHOT_DATE
    hour = hour or SNAPSHOT_HOUR
    fname = f"bview.{collector}.{date}.{hour}.gz"
    fpath = DATA_DIR / f"{fname}"
    if fpath.exists():
        print(f"Found in cache: {fname}")
    else:
        fpath.parent.mkdir(parents=True, exist_ok=True)
        download_url = (
            f"https://data.ris.ripe.net/"
            f"{collector}/{date[:4]}.{date[4:6]}/bview.{date}.{hour}.gz"
        )
        with requests.get(download_url, stream=True) as resp:
            resp.raise_for_status()
            with open(fpath, "wb") as fout:
                shutil.copyfileobj(resp.raw, fout)
        print(f"Downloaded: {fname}")
    return fpath

RIB_PATHS = [ fetch_rib_file(c) for c in COLLECTORS ]
print("Fetched all RIB files.")

### Task 1.2: Enrich ASN data with organization names

In [ ]:
ASNAMES_URL = "https://ftp.ripe.net/ripe/asnames/asn.txt"
fname = "asnames.txt"
fpath = DATA_DIR / f"{fname}"

if not fpath.exists():
    with requests.get(ASNAMES_URL, stream=True) as resp:
        resp.raise_for_status()
        with open(fpath, "w", encoding="utf-8") as fout:
            fout.write(resp.text)
    print(f"Downloaded: {fname}")
    
else:
    try:
        data = {}
        with open(fpath, "r", encoding="utf-8") as fin:
            for line in fin:
                asn, _ , rest = line.strip().partition(" ")
                name, _ , country = rest.rpartition(", ")
                data[asn] = name
    except Exception as e: 
        print(f"An error occurred: {e}")

# store ASN-name mapping in a dataframe
asn_info_df = pd.DataFrame.from_dict(data, orient='index', columns=['name'])
asn_info_df.head(10)

## Task 2: Convert RIB data into a weighted graph

Conceptually, BC and hegemony treat ASes and AS paths as vertices and edges in a graph. In this task, you will process the RIB files that you just fetched, and extract AS path information for later use. 

You might want to refer to the [bgpkit documentation](https://docs.rs/bgpkit-parser/latest/bgpkit_parser).

`TODO`: if two peers are seeing the same AS path, then we might not want to double count them. 

### Task 2.1: Compute Address Space Size

This function is based on `count_addresses_per_asn(ipv4_single_origin)` in the BGP assignment. 

The main change is that we count the number of addresses within a prefix, not within an ASN. We call the number of addresses the weight of a given prefix. 

**TODO**: The main change is that we return two dictionaries: one is keyed by raw prefix strings, and the other is keyed by normalized prefix strings. This style of implementation has the advantage of avoiding having to parse the raw prefixes twice later on in the notebook.

In [ ]:
def get_address_count(pfx):
    pfx_len = 32 - int(pfx.split("/")[1])
    return 1 << pfx_len
    
def weigh_prefixes(pyt):
    """
    Assign weights to each observed prefix by the size of its address space.
    No double counting (see task description). 
    """
    pfx_to_weight = {}
    
    for pfx in pyt:
        pfx_weight = get_address_count(pfx)
        children_weight = 0
        for child in pyt.children(pfx):
            if pyt.parent(child) == pfx:
                children_weight += get_address_count(child)
        pfx_to_weight[pfx] = pfx_weight - children_weight
        
    return pfx_to_weight

### Task 2.2: Build Local Graphs

In [ ]:
def build_local_graphs(RIB_PATHS):
    """
    Each element returned by bgpkit parser represents a RIB table entry with the
    following format. For example:
    {
        'elem_type': 'R', 
        'peer_asn': 1234,
        'peer_address': '80.77.16.114',
        'as-path': '1234 956 14068',
        'origin': 14068,
        'prefix': '216.163.136.0/24'
    }
    
    We define 'full_path' as an AS path from peer's ASN to origin ASN. 
    We define 'atom' as an AS path excluding its origin AS.
    
    TODO: Probably need to convert nested defaultdicts in local_graphs
    to regular dicts. 
    TODO: Should ignore MOAS prefixes or somehow deal with it.
    """
    pyt = pytricia.PyTricia(32)            # radix tree for prefix weighing later
    local_graphs = defaultdict(lambda: defaultdict(set))  # single peer views
    peer_to_asn = defaultdict(set)         # map a peer to every ASN it observes
    asn_to_peer = defaultdict(set)         # map an ASN to every peer that sees it
    n_processed = 0                        # progress report 

    for p in RIB_PATHS:
        for element in bgpkit.Parser(url=str(p)):
            if element.elem_type != "A": # only process route announcements
                continue
            origin_pfx = element.prefix
            if ":" in origin_pfx or origin_pfx.endswith("/0"): # skip v6 and default routes
                continue
            
            pyt.insert(origin_pfx, 0)

            full_path = element.as_path.split(" ")
            atom = tuple(full_path[:-1])
            origin_asn = full_path[0]
            
            # update VP's view
            peer_ip = element.peer_ip
            local_graphs[peer_ip][atom].add((origin_asn, origin_pfx))

            peer_to_asn[peer_ip].add(origin_asn)
            asn_to_peer[origin_asn].add(peer_ip)
            
            n_processed += 1
            if n_processed % 5_000_000 == 0: 
                print(f"...Processed {n_processed:,} entries")
                
    print(f"\nProcessed {n_processed:,} RIB entries across {len(RIB_PATHS):,} snapshot(s).")
    return pyt, dict(local_graphs), dict(peer_to_asn), dict(asn_to_peer)

### Task 2.3: Build Global Graph

In [ ]:
def _iter_local_graphs(local_graphs, peer=None):
    """
    TODO: Maybe we want to describe what 'yield' does.
    """
    if peer:
        graphs = {peer: local_graphs[peer]}
    else:
        graphs = local_graphs

    for peer_ip, atom_to_origins in graphs.items():
        for atom, origin_asn_pfx_pairs in atom_to_origins.items():
            for origin_asn, origin_pfx in origin_asn_pfx_pairs:
                yield peer_ip, atom, origin_asn, origin_pfx

In [ ]:
def build_global_graph(pfx_to_weight, local_graphs):
    """
    Now, we merge the local graphs observed by each VP into a global graph. 
    TODO: Finish function description. esp the use of 'global graph'. 
    TODO: I'm probably inflating the count of unique paths by including origin_pfx. 
    
    E.g., we consider the following as two different paths: 
    A -> B -> C (10.0.1.0/24)
    A -> B -> C (10.0.2.0/24)
    """
    unique_paths = set()
    global_graph = defaultdict(lambda:{
        "uw": 0, "w": 0,
        "bc_uw": 0.0, "bc_w": 0.0,
        "hege_uw": 0.0, "hege_w": 0.0
    })
    
    for peer_ip, atom, origin_asn, origin_pfx in _iter_local_graphs(local_graphs):
        # process unique paths only
        path = (atom, origin_asn, origin_pfx)
        if path in unique_paths:
            continue
        unique_paths.add(path)
        
        # assign weight to every ASN along the path, including origin ASN
        weight = pfx_to_weight[origin_pfx]
        global_graph[origin_asn]["uw"] += 1
        global_graph[origin_asn]["w"] += weight
        for asn in atom:
            global_graph[asn]["uw"] += 1
            global_graph[asn]["w"] += weight
            
    return unique_paths, dict(global_graph)

In [ ]:
pyt, local_graphs, peer_to_asn, asn_to_peer = build_local_graphs(RIB_PATHS)
pfx_to_weight = weigh_prefixes(pyt)
unique_paths, global_graph = build_global_graph(pfx_to_weight, local_graphs)
      
n_prefix = len(pfx_to_weight)
n_path = len(unique_paths)
n_peer = len(peer_to_asn)
n_asn = len(asn_to_peer)
observed_addrs = sum(pfx_to_weight.values())

print(f"Observed {n_asn:,} ASNs.")
print(f"Observed {n_path:,} unique AS paths.")
print(f"Observed IPv4 address space: {observed_addrs:,} addresses "
      f"by {n_peer:,} peers from collector(s): {', '.join(COLLECTORS)}.")
print(f"Weighted {n_prefix:,} prefixes.")

## Task 3: Betweenness Centrality

In [ ]:
def get_bc_score_global(global_graph, n_path):
    """
    TODO: The way I'm calculating bc_w is definitely wrong. 
    The values for weighted BC looks too high. 
    """
    total_path_weight = sum(pfx_to_weight[pfx] for _, _, pfx in unique_paths)
    n_path = len(unique_paths)
    for _, stats in global_graph.items():
        stats["bc_uw"] = stats["uw"] / n_path
        stats["bc_w"] = stats["w"] / total_path_weight

get_bc_score_global(global_graph, n_path)

In [ ]:
global_graph_df = pd.DataFrame.from_dict(global_graph, orient="index")

global_graph_enriched = (
    global_graph_df
    .merge(asn_info_df, left_index=True, right_index=True)
    .sort_values("bc_w", ascending=False)
    .reset_index(names="asn")
    .assign(
        bc_uw_rank=lambda df: df['bc_uw'].rank(ascending=False).astype(int),
        bc_w_rank=lambda df: df['bc_w'].rank(ascending=False).astype(int)
    )
    .reindex(columns=["asn","name","bc_uw","bc_w", "bc_uw_rank","bc_w_rank"])
)

global_graph_enriched.sort_values("bc_uw", ascending=False).head(10)

### Question 1
What does betweenness centrality measure?

YOUR ANSWER HERE

### Question 2
Why would we use the weighted betweenness centrality versus the unweighted betweenness centrality?

YOUR ANSWER HERE

### Question 3
What are the top 5 ASes ranked by their weighted betweenness centralities? How these compare to the top ASes [as ranked by customer cone size](https://asrank.caida.org/)? Does this make sense? Why?

YOUR ANSWER HERE

### Task 3.1: Distribution of betweenness centrality across ASes

In [ ]:
"""
CCDF for unweighted betweenness centrality.
TODO: add some hints. 
"""
bc_uw_list = [v for v in global_graph_df["bc_uw"] if v > 0]
bc_uw_dist = Counter(bc_uw_list)
x_bc_uw = sorted(bc_uw_dist.keys())
y_bc_uw = []
remaining = len(bc_uw_list)
for xi in x_bc_uw:
    y_bc_uw.append(remaining)
    remaining -= bc_uw_dist[xi]

"""
CCDF for weighted betweenness centrality.
TODO: add some hints.
"""
bc_w_list = [v for v in global_graph_df["bc_w"] if v > 0]
bc_w_dist = Counter(bc_w_list)
x_bc_w = sorted(bc_w_dist.keys())
y_bc_w = []
remaining = len(bc_w_list)
for xi in x_bc_w:
    y_bc_w.append(remaining)
    remaining -= bc_w_dist[xi]

fig, ax1 = plt.subplots()
ax1.plot(x_bc_uw, y_bc_uw, marker=".", markersize=5, color="red", label="unweighted")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Betweenness centrality (unweighted)")
ax1.set_ylabel("Number of ASes")

ax2 = ax1.twiny()
ax2.plot(x_bc_w, y_bc_w, marker=".", markersize=5, color="blue", label="weighted")
ax2.set_xscale("log")
ax2.set_xlabel("Betweenness centrality (weighted)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
ax1.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

### Question 4
What shape does the CCDF have? What does this tell us about the distribution of weighted betweenness centralities?

YOUR ANSWER HERE

## Task 4: AS Hegemony

In this task you will:

1. Compute $BC_{(j)}(v)$ for each viewpoint $j$ and AS $v$;
2. Filter out VPs and compute quotients to calculate AS hegemony;
3. Create a scatter plot of AS hegemony vs. weighted betweenness centrality;
4. Answer some questions about the data.

In [ ]:
def compute_one(n, n_drop):
    """
    TODO: Add description. 
    """
    bc_score_lst = []
    bc_score_lst.extend( [0] * len(n) )
    if n_drop > 0:
        bc_score_lst = bc_score_lst[n_drop:-n_drop]

    n_unbiased_vp = len(bc_score_lst)
    if n_unbiased_vp > 0:
        hege = sum(bc_score_lst) / n_unbiased_vp
    else:
        hege = 0.0
    
    return hege

In [ ]:
# def get_hegemony_scores(asn_to_peer, peer_to_asn, local_graphs, pfx_to_weight):
#     """
#     Drop the top 10 percent and bottom 10 percent of viewpoints. 
#     """
#     nproc = 20
#     ALPHA = 0.1
#     all_asn = set(asn_to_peer.keys())
#     all_peer = set(peer_to_asn.keys())
#     n_drop = int(len(all_peer) * ALPHA)

#     hege_score_uw = defaultdict(float)
#     hege_score_w = defaultdict(float)

#     for target_asn in all_asn:
#         bc_score_lst_uw = []
#         bc_score_lst_w = []
#         for peer in all_peer:
#             """
#             For every peer, compute its BC score for target_asn.
#             (1) If this peer observes target_asn, let it score target_asn. 
#             (2) If it DOES NOT observe target_asn, it gives target_asn a zero.
#             """
#             path_with_asn_uw = 0
#             path_with_asn_w = 0
#             total_observed_path_uw = 0
#             total_observed_path_w = 0
#             for _, atom, o_asn, o_pfx in _iter_local_graphs(local_graphs, peer):
#                 """
#                 For each path observed by the given peer:
#                 (1) Weight path by its origin AS's address space size. 
#                 (2) Check if target_asn is observed in this path. 
#                 (3) Assign BC score to target_asn. 
#                 """
#                 weight = pfx_to_weight[o_pfx]
                
#                 total_observed_path_uw += 1
#                 total_observed_path_w += weight

#                 if target_asn in atom or target_asn == o_asn:
#                     path_with_asn_uw +=1 
#                     path_with_asn_w += weight

#             bc_score_uw = path_with_asn_uw / total_observed_path_uw
#             bc_score_w = path_with_asn_w / total_observed_path_w

#             bc_score_lst_uw.append(bc_score_uw)
#             bc_score_lst_w.append(bc_score_w)

#         observing_peers = asn_to_peer[target_asn]
#         n_non_observing_peers = len(all_peer - observing_peers)
        
#         bc_score_lst_uw.extend( [0] * n_non_observing_peers )
#         bc_score_lst_uw.sort()
#         bc_score_lst_w.extend( [0] * n_non_observing_peers )
#         bc_score_lst_w.sort()
        
#         if n_drop > 0:
#             bc_score_lst_uw = bc_score_lst_uw[n_drop:-n_drop]
#             bc_score_lst_w = bc_score_lst_w[n_drop:-n_drop]

#         hege_score_uw[target_asn] = sum(bc_score_lst_uw) / len(bc_score_lst_uw)
#         hege_score_w[target_asn] = sum(bc_score_lst_w) / len(bc_score_lst_w)

#     return hege_score_uw, hege_score_w

# hege_uw, hege_w = get_hegemony_scores(asn_to_peer, peer_to_asn, local_graphs, pfx_to_weight)

In [ ]:
def _init_worker(local_graphs, pfx_to_weight, asn_to_peer, all_peer, n_drop):
    global _local_graphs, _pfx_to_weight, _asn_to_peer, _all_peer, _n_drop
    _local_graphs = local_graphs
    _pfx_to_weight = pfx_to_weight
    _asn_to_peer = asn_to_peer
    _all_peer = all_peer
    _n_drop = n_drop

def _hegemony_for_asn(target_asn):
    bc_score_lst_uw = []
    bc_score_lst_w = []

    for peer in _all_peer:
        if peer not in _local_graphs:
            continue
        path_with_asn_uw = 0
        path_with_asn_w = 0
        total_observed_path_uw = 0
        total_observed_path_w = 0
        for _, atom, o_asn, o_pfx in _iter_local_graphs({peer: _local_graphs[peer]}):
            weight = _pfx_to_weight[o_pfx]
            total_observed_path_uw += 1
            total_observed_path_w += weight
            if target_asn in atom or target_asn == o_asn:
                path_with_asn_uw += 1
                path_with_asn_w += weight
        if total_observed_path_uw == 0:
            continue
        bc_score_lst_uw.append(path_with_asn_uw / total_observed_path_uw)
        bc_score_lst_w.append(path_with_asn_w / total_observed_path_w)

    observing_peers = _asn_to_peer[target_asn]
    n_non_observing_peers = len(_all_peer - observing_peers)

    bc_score_lst_uw.extend([0] * n_non_observing_peers)
    bc_score_lst_uw.sort()
    bc_score_lst_w.extend([0] * n_non_observing_peers)
    bc_score_lst_w.sort()

    if _n_drop > 0:
        bc_score_lst_uw = bc_score_lst_uw[_n_drop:-_n_drop]
        bc_score_lst_w = bc_score_lst_w[_n_drop:-_n_drop]

    h_uw = sum(bc_score_lst_uw) / len(bc_score_lst_uw)
    h_w = sum(bc_score_lst_w) / len(bc_score_lst_w)
    return target_asn, h_uw, h_w


def get_hegemony_scores(asn_to_peer, peer_to_asn, local_graphs, pfx_to_weight, nproc=20):
    """
    Drop the top 10 percent and bottom 10 percent of viewpoints.
    """
    ALPHA = 0.1
    all_asn = set(asn_to_peer.keys())
    all_peer = set(peer_to_asn.keys())
    n_drop = int(len(all_peer) * ALPHA)

    hege_score_uw = defaultdict(float)
    hege_score_w = defaultdict(float)

    with mp.Pool(
        nproc,
        initializer=_init_worker,
        initargs=(local_graphs, pfx_to_weight, asn_to_peer, all_peer, n_drop),
    ) as pool:
        for target_asn, h_uw, h_w in pool.imap_unordered(_hegemony_for_asn, all_asn):
            hege_score_uw[target_asn] = h_uw
            hege_score_w[target_asn] = h_w

    return hege_score_uw, hege_score_w

hege_uw, hege_w = get_hegemony_scores(asn_to_peer, peer_to_asn, local_graphs, pfx_to_weight)

In [ ]:
hege_df = (
    pd.DataFrame.from_dict(hege_uw, orient="index", columns=["h_score_uw"])
    .reset_index(names="asn")
    .sort_values("h_score_uw", ascending=False)
)
hege_df.head(10)

### Question 1
Why do we remove the top and bottom $\alpha$ proportions of the viewpoints?

YOUR ANSWER HERE

### Question 2
How do the top 5 ASes ranked by betweenness centrality compare to the top 5 ranked by AS hegemony?

YOUR ANSWER HERE

In [ ]:
both = df.merge(hege_df, on="asn", suffixes=("_bc", "_hege"))
pos = both[(both["bc_weighted"] > 0) & (both["hegemony"] > 0)]

fig, ax = plt.subplots(figsize=(6.5, 6))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

lo = max(min(pos["bc_weighted"].min(), pos["hegemony"].min()), 1e-9)
hi = max(pos["bc_weighted"].max(), pos["hegemony"].max()) * 2
ax.plot([lo, hi], [lo, hi], color=MUTED, linewidth=1, linestyle="--", zorder=1)
ax.annotate("equal under both metrics", xy=(hi, hi),
            xytext=(-8, -14), textcoords="offset points",
            ha="right", fontsize=8, color=MUTED)

ax.scatter(pos["bc_weighted"], pos["hegemony"],
           s=14, color=BLUE, alpha=0.45, linewidths=0, zorder=2)

asn_outlier = pos.iloc[(pos["bc_weighted"] / pos["hegemony"]).argmax()]
ax.annotate(f"AS{asn_outlier['asn']}", xy=(asn_outlier["bc_weighted"], asn_outlier["hegemony"]),
            xytext=(5, 3), textcoords="offset points",
            fontsize=8, color=INK2)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Weighted betweenness centrality", color=INK2)
ax.set_ylabel("AS hegemony", color=INK2)
fig.tight_layout()
plt.show()

### Question 3
What does the concentration of points on the $x = y$ line tell us about the relationship between betweenness centrality and AS hegemony?

YOUR ANSWER HERE

### Question 4
Look up the AS that corresponds to the outlier point. How does this AS's geographic location relate to the location of the collectors?

YOUR ANSWER HERE

### Question 5
Why does the geographic location of the outlier AS cause its betweenness centrality to be so much larger than its AS hegemony?

YOUR ANSWER HERE